# PDF → LaTeX Transformer — Full Training Pipeline

This notebook integrates the complete **PDF2LaTeX** training pipeline end-to-end:

| # | Section | Description |
|---|---------|-------------|
| 1 | **Setup** | Install deps, configure paths |
| 2 | **Config** | Inline YAML configs |
| 3 | **Preprocessing** | Tokeniser training & PDF rendering |
| 4 | **Dataset** | Dataset classes & DataLoaders |
| 5 | **Models** | Vision Encoder, LaTeX Decoder, PDF2LaTeX |
| 6 | **Metrics** | BLEU, edit distance, format score, compilation |
| 7 | **Supervised Pre-training** | Teacher-forced cross-entropy loop |
| 8 | **Reward Functions** | Compilation, similarity, format rewards |
| 9 | **Rollout Engine** | Sampling + log-prob collection |
| 10 | **PPO Fine-tuning** | Proximal Policy Optimisation |
| 11 | **GRPO Fine-tuning** | Group Relative Policy Optimisation (recommended) |
| 12 | **Reward Model** | Optional learned reward from human preferences |
| 13 | **Inference** | Run the trained model on new PDFs |
| 14 | **Evaluation** | Full benchmark on a test set |

> **Data layout expected:**
> ```
> data/train/   sample_001.pdf  sample_001.tex  ...
> data/val/     sample_002.pdf  sample_002.tex  ...
> data/test/    ...
> ```


## 1 · Setup

In [ ]:
# Install required packages (run once)
%pip install -q torch torchvision transformers tokenizers pymupdf \
               Pillow tqdm pyyaml editdistance accelerate

# Optional: experiment tracking
# %pip install -q wandb

import sys, os
from pathlib import Path

# If running from the repo root, add it to sys.path so local modules resolve
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Python:", sys.version)
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


## 2 · Configuration

All hyper-parameters live in two dicts — `TRAIN_CFG` and `FINETUNE_CFG`. Adjust before running.

In [ ]:
TRAIN_CFG = {
    "data": {
        "train_dir":  "data/train",
        "val_dir":    "data/val",
        "image_size": 896,
        "max_seq_len": 4096,
        "num_workers": 0,          # set >0 if you have multiple CPUs
        "pdf_dpi":    150,
    },
    "model": {
        "patch_size":   16,
        "encoder_dim":  768,
        "encoder_layers": 12,
        "encoder_heads":  12,
        "encoder_mlp_ratio": 4.0,
        "encoder_dropout":   0.1,
        "decoder_dim":   768,
        "decoder_layers": 12,
        "decoder_heads":  12,
        "decoder_mlp_ratio": 4.0,
        "decoder_dropout":   0.1,
        "vocab_size":  8192,
        "max_position_embeddings": 4096,
        # Set to None to train from scratch:
        "vision_backbone":    "google/vit-base-patch16-224",
        "init_from_pretrained": True,
    },
    "training": {
        "epochs":   50,
        "batch_size": 8,
        "gradient_accumulation_steps": 4,
        "learning_rate": 3e-4,
        "warmup_steps":  2000,
        "weight_decay":  0.01,
        "max_grad_norm": 1.0,
        "lr_scheduler":  "cosine",
        "label_smoothing": 0.1,
        "fp16": False,
        "bf16": True,
        "checkpoint_dir": "checkpoints/pretrain",
        "save_every_n_steps": 1000,
        "keep_last_n_checkpoints": 3,
        "log_every_n_steps": 50,
        "use_wandb": False,
        "wandb_project": "pdf2latex",
        "wandb_run_name": "pretrain-v1",
        "val_every_n_steps": 500,
        "val_max_batches":   50,
    },
}

FINETUNE_CFG = {
    **TRAIN_CFG,   # inherits data / model blocks
    "model": {**TRAIN_CFG["model"], "checkpoint": "checkpoints/pretrain/best.pt"},
    "rollout": {
        "max_new_tokens":        4096,
        "temperature":           0.8,
        "top_p":                 0.95,
        "num_samples_per_prompt": 4,    # GRPO group size G
        "rollout_batch_size":     4,
    },
    "reward": {
        "compilation_weight": 0.5,
        "similarity_weight":  0.3,
        "format_weight":      0.2,
        "latex_compiler":  "pdflatex",
        "compiler_timeout": 30,
    },
    "ppo": {
        "epochs":        10,
        "clip_epsilon":  0.2,
        "value_loss_coef": 0.5,
        "entropy_coef":    0.01,
        "kl_coef":         0.1,
        "target_kl":       0.05,
        "gamma":  1.0,
        "lam":    0.95,
        "critic_lr": 1e-4,
    },
    "grpo": {
        "kl_coef":             0.04,
        "clip_epsilon":        0.2,
        "normalize_advantages": True,
    },
    "training": {
        **TRAIN_CFG["training"],
        "method":       "grpo",   # "ppo" or "grpo"
        "total_steps":  5000,
        "policy_lr":    5e-6,
        "warmup_steps": 100,
        "weight_decay": 0.0,
        "max_grad_norm": 1.0,
        "bf16":          True,
        "gradient_accumulation_steps": 1,
        "checkpoint_dir":      "checkpoints/finetune",
        "save_every_n_steps":  500,
        "log_every_n_steps":   10,
        "use_wandb": False,
        "wandb_project": "pdf2latex",
        "wandb_run_name": "grpo-v1",
    },
}

print("✓ Configs ready")


## 3 · Preprocessing

### 3.1 LaTeX BPE Tokeniser
Train a byte-level BPE tokeniser on all `.tex` files in your corpus.


In [ ]:
from __future__ import annotations
import os, re, random, glob, json, math, time, copy, subprocess, tempfile
from pathlib import Path
from typing import Optional
from collections import Counter

# ── Tokeniser ─────────────────────────────────────────────────────────────────

def build_latex_tokenizer(vocab_size: int = 8192, corpus_files=None):
    """BPE tokeniser trained on .tex files (or minimal fallback)."""
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

    tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]

    if corpus_files:
        trainer = trainers.BpeTrainer(
            vocab_size=vocab_size, special_tokens=special_tokens, min_frequency=2)
        tokenizer.train(corpus_files, trainer)
    else:
        tokenizer.add_special_tokens(special_tokens)

    return tokenizer

def train_tokenizer_from_corpus(corpus_root: str, vocab_size: int = 8192,
                                output_path: str = "tokenizer.json"):
    """Train and save a tokeniser from all .tex files under corpus_root."""
    tex_files = glob.glob(str(Path(corpus_root) / "**" / "*.tex"), recursive=True)
    print(f"Found {len(tex_files)} .tex files")
    tok = build_latex_tokenizer(vocab_size=vocab_size, corpus_files=tex_files or None)
    tok.save(output_path)
    print(f"Tokeniser saved → {output_path}")
    return tok

def load_tokenizer(path: str):
    from tokenizers import Tokenizer
    return Tokenizer.from_file(path)

# Usage (run if you have a .tex corpus):
# tokenizer = train_tokenizer_from_corpus("data/train", vocab_size=8192)

# For this notebook we build a minimal fallback tokeniser:
tokenizer = build_latex_tokenizer(vocab_size=256)
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")
PAD_ID = tokenizer.token_to_id("<pad>")
print(f"BOS={BOS_ID}  EOS={EOS_ID}  PAD={PAD_ID}")
print(f"Vocab size: {tokenizer.get_vocab_size()}")


### 3.2 PDF Rendering & Page Splitting

In [ ]:
from PIL import Image

try:
    import fitz   # PyMuPDF
    HAS_FITZ = True
except ImportError:
    HAS_FITZ = False
    print("PyMuPDF not found — install with: pip install pymupdf")

def render_pdf_page(pdf_path, page_idx: int = 0, dpi: int = 150) -> Image.Image:
    """Render a single PDF page to a PIL RGB image."""
    if not HAS_FITZ:
        raise RuntimeError("PyMuPDF (fitz) required: pip install pymupdf")
    doc = fitz.open(str(pdf_path))
    page = doc[page_idx]
    zoom = dpi / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), colorspace=fitz.csRGB)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    doc.close()
    return img

def count_pdf_pages(pdf_path) -> int:
    if not HAS_FITZ:
        raise RuntimeError("PyMuPDF required")
    doc = fitz.open(str(pdf_path))
    n = doc.page_count
    doc.close()
    return n

def split_latex_by_page(tex_source: str) -> list:
    """Split .tex source on page-break commands."""
    source = tex_source.replace("\r\n", "\n")
    parts = re.split(r"\\(?:newpage|clearpage|pagebreak)\b", source)
    parts = [p.strip() for p in parts if p.strip()]
    return parts if parts else [source]

# Quick self-test
sample_tex = r"\section{Intro}\nHello\n\newpage\n\section{Methods}"
chunks = split_latex_by_page(sample_tex)
print(f"Split into {len(chunks)} chunks:", [c[:30] for c in chunks])


### 3.3 Pre-render PDFs to PNG cache (optional speedup)

In [ ]:
def prerender_dataset(data_dir: str, output_dir: str, dpi: int = 150, image_size: int = 896):
    """
    Render all PDFs to PNG and write a manifest JSON.
    Skips if output_dir/manifest.json already exists.
    """
    data_dir   = Path(data_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "manifest.json"
    if manifest_path.exists():
        print(f"Cache already exists at {output_dir}. Delete to regenerate.")
        return json.loads(manifest_path.read_text())

    manifest = {}
    for pdf_path in sorted(data_dir.glob("*.pdf")):
        doc = fitz.open(str(pdf_path))
        for page_idx in range(doc.page_count):
            page = doc[page_idx]
            zoom = dpi / 72.0
            pix  = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), colorspace=fitz.csRGB)
            img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            img  = img.resize((image_size, image_size), Image.LANCZOS)
            name = f"{pdf_path.stem}_page{page_idx}.png"
            img.save(str(output_dir / name))
            manifest[f"{pdf_path.stem}::{page_idx}"] = str(output_dir / name)
        doc.close()

    manifest_path.write_text(json.dumps(manifest, indent=2))
    print(f"Pre-rendered {len(manifest)} pages → {output_dir}")
    return manifest

# Usage:
# prerender_dataset("data/train", "data/train_cache")
print("prerender_dataset() ready — call with your data dir to pre-cache PDFs.")


## 4 · Dataset & DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def build_image_transform(image_size: int = 896):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std =[0.229, 0.224, 0.225]),
    ])

class PDFLatexDataset(Dataset):
    """
    Paired (PDF-page image, LaTeX chunk) dataset.

    root/
      sample_001.pdf   sample_001.tex
      sample_002.pdf   sample_002.tex  ...
    """
    def __init__(self, root, tokenizer, image_transform,
                 max_seq_len=4096, dpi=150, augment=False):
        self.root = Path(root)
        self.tokenizer = tokenizer
        self.image_transform = image_transform
        self.max_seq_len = max_seq_len
        self.dpi = dpi
        self.augment = augment
        self.samples = []
        self._build_index()

    def _build_index(self):
        pdf_files = sorted(self.root.glob("*.pdf"))
        if not pdf_files:
            raise FileNotFoundError(f"No .pdf files in {self.root}")
        for pdf_path in pdf_files:
            tex_path = pdf_path.with_suffix(".tex")
            if not tex_path.exists():
                continue
            tex_source = tex_path.read_text(encoding="utf-8", errors="replace")
            chunks     = split_latex_by_page(tex_source)
            try:
                n_pages = count_pdf_pages(pdf_path)
            except Exception:
                n_pages = 1
            for i in range(n_pages):
                self.samples.append((pdf_path, i, chunks[i % len(chunks)]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pdf_path, page_idx, tex_chunk = self.samples[idx]
        img = render_pdf_page(pdf_path, page_idx, self.dpi)
        if self.augment:
            img = self._augment(img)
        pixel_values = self.image_transform(img)

        bos, eos, pad = BOS_ID, EOS_ID, PAD_ID
        ids = [bos] + self.tokenizer.encode(tex_chunk).ids + [eos]
        if len(ids) > self.max_seq_len:
            ids = ids[:self.max_seq_len - 1] + [eos]
        seq_len = len(ids)
        padding = self.max_seq_len - seq_len
        input_ids      = ids + [pad] * padding
        attention_mask = [1] * seq_len + [0] * padding
        labels = input_ids[1:] + [-100]
        labels = [l if m == 1 else -100 for l, m in zip(labels, attention_mask)]

        return {
            "pixel_values":   pixel_values.float(),
            "input_ids":      torch.tensor(input_ids,      dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels":         torch.tensor(labels,         dtype=torch.long),
            "tex_chunk":      tex_chunk,
        }

    def _augment(self, img):
        from PIL import ImageFilter, ImageEnhance
        if random.random() < 0.3:
            img = img.rotate(random.uniform(-2, 2), fillcolor=(255,255,255))
        if random.random() < 0.5:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))
        if random.random() < 0.3:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0, 0.8)))
        return img


def build_dataloaders(cfg, tokenizer):
    img_tf = build_image_transform(cfg["data"]["image_size"])
    train_ds = PDFLatexDataset(cfg["data"]["train_dir"], tokenizer, img_tf,
                               cfg["data"]["max_seq_len"], cfg["data"]["pdf_dpi"], augment=True)
    val_ds   = PDFLatexDataset(cfg["data"]["val_dir"],   tokenizer, img_tf,
                               cfg["data"]["max_seq_len"], cfg["data"]["pdf_dpi"], augment=False)
    train_loader = DataLoader(train_ds, batch_size=cfg["training"]["batch_size"],
                              shuffle=True,  num_workers=cfg["data"]["num_workers"],
                              pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["training"]["batch_size"],
                              shuffle=False, num_workers=cfg["data"]["num_workers"],
                              pin_memory=True)
    return train_loader, val_loader

print("Dataset classes ready.")
print("Usage: train_loader, val_loader = build_dataloaders(TRAIN_CFG, tokenizer)")


## 5 · Model Definitions
### 5.1 Vision Encoder (ViT-style)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# ── Patch embedding ────────────────────────────────────────────────────────────
class PatchEmbedding(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, embed_dim):
        super().__init__()
        assert image_size % patch_size == 0
        self.num_patches = (image_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)   # B × N × D

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
    def forward(self, x, attn_mask=None):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = self.attn_drop(attn.softmax(dim=-1))
        return self.proj_drop(self.proj((attn @ v).transpose(1,2).reshape(B, N, C)))

class EncoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = MultiHeadSelfAttention(dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(nn.Linear(dim, mlp_hidden), nn.GELU(),
                                   nn.Dropout(dropout), nn.Linear(mlp_hidden, dim),
                                   nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class VisionEncoder(nn.Module):
    """ViT-style vision encoder: B×3×H×W → B×N×D (patch embeddings)."""
    def __init__(self, image_size=896, patch_size=16, in_channels=3,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.embed_dim   = embed_dim
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        N = self.patch_embed.num_patches
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.zeros(1, N + 1, embed_dim))
        self.pos_drop    = nn.Dropout(dropout)
        self.blocks      = nn.ModuleList([EncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
                                          for _ in range(depth)])
        self.norm        = nn.LayerNorm(embed_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B = x.size(0)
        x = torch.cat([self.cls_token.expand(B,-1,-1), self.patch_embed(x)], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)[:, 1:, :]   # drop CLS → B × N × D

class HFVisionEncoder(nn.Module):
    """Pretrained HuggingFace ViT wrapper."""
    def __init__(self, model_name="google/vit-base-patch16-224", output_dim=768):
        super().__init__()
        from transformers import ViTModel
        self.vit = ViTModel.from_pretrained(model_name)
        vit_dim = self.vit.config.hidden_size
        self.proj = nn.Linear(vit_dim, output_dim) if vit_dim != output_dim else nn.Identity()
        self.embed_dim = output_dim
    def forward(self, x):
        return self.proj(self.vit(pixel_values=x).last_hidden_state[:, 1:, :])

def build_encoder(cfg):
    m = cfg["model"]
    if m.get("init_from_pretrained") and m.get("vision_backbone"):
        return HFVisionEncoder(m["vision_backbone"], m["encoder_dim"])
    return VisionEncoder(image_size=cfg["data"]["image_size"], patch_size=m["patch_size"],
                         embed_dim=m["encoder_dim"], depth=m["encoder_layers"],
                         num_heads=m["encoder_heads"], mlp_ratio=m["encoder_mlp_ratio"],
                         dropout=m["encoder_dropout"])

print("VisionEncoder defined ✓")


### 5.2 Autoregressive LaTeX Decoder

In [ ]:
def _sample_top_p(logits, temperature, top_p):
    logits = logits / max(temperature, 1e-8)
    probs  = F.softmax(logits, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, dim=-1, descending=True)
    cumulative = sorted_probs.cumsum(dim=-1)
    sorted_probs[cumulative - sorted_probs > top_p] = 0.0
    sorted_probs.div_(sorted_probs.sum(dim=-1, keepdim=True))
    next_token = torch.multinomial(sorted_probs, num_samples=1)
    return sorted_idx.gather(dim=-1, index=next_token)

class CausalSelfAttention(nn.Module):
    def __init__(self, dim, num_heads, dropout=0.0, max_seq_len=4096):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv  = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1)
        self.register_buffer("causal_mask", mask.bool())
    def forward(self, x, key_padding_mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B,T,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        attn = attn.masked_fill(self.causal_mask[:T,:T].unsqueeze(0).unsqueeze(0), float("-inf"))
        if key_padding_mask is not None:
            attn = attn.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2), float("-inf"))
        attn = self.attn_drop(attn.softmax(dim=-1))
        return self.proj_drop(self.proj((attn @ v).transpose(1,2).reshape(B,T,C)))

class CrossAttention(nn.Module):
    def __init__(self, decoder_dim, encoder_dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = decoder_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.q_proj   = nn.Linear(decoder_dim, decoder_dim)
        self.k_proj   = nn.Linear(encoder_dim,  decoder_dim)
        self.v_proj   = nn.Linear(encoder_dim,  decoder_dim)
        self.out_proj = nn.Linear(decoder_dim, decoder_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
    def forward(self, x, enc, enc_padding_mask=None):
        B, T, _ = x.shape;  S = enc.size(1)
        q = self.q_proj(x).reshape(B,T,self.num_heads,self.head_dim).transpose(1,2)
        k = self.k_proj(enc).reshape(B,S,self.num_heads,self.head_dim).transpose(1,2)
        v = self.v_proj(enc).reshape(B,S,self.num_heads,self.head_dim).transpose(1,2)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        if enc_padding_mask is not None:
            attn = attn.masked_fill(enc_padding_mask.unsqueeze(1).unsqueeze(2), float("-inf"))
        attn = self.attn_drop(attn.softmax(dim=-1))
        return self.proj_drop(self.out_proj((attn @ v).transpose(1,2).reshape(B,T,-1)))

class DecoderBlock(nn.Module):
    def __init__(self, decoder_dim, encoder_dim, num_heads, mlp_ratio=4.0,
                 dropout=0.0, max_seq_len=4096):
        super().__init__()
        self.norm1     = nn.LayerNorm(decoder_dim)
        self.self_attn = CausalSelfAttention(decoder_dim, num_heads, dropout, max_seq_len)
        self.norm2     = nn.LayerNorm(decoder_dim)
        self.cross_attn = CrossAttention(decoder_dim, encoder_dim, num_heads, dropout)
        self.norm3     = nn.LayerNorm(decoder_dim)
        mlp_hidden = int(decoder_dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(decoder_dim, mlp_hidden), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(mlp_hidden, decoder_dim),
                                 nn.Dropout(dropout))
    def forward(self, x, enc, tgt_key_padding_mask=None, enc_padding_mask=None):
        x = x + self.self_attn(self.norm1(x), tgt_key_padding_mask)
        x = x + self.cross_attn(self.norm2(x), enc, enc_padding_mask)
        return x + self.mlp(self.norm3(x))

class LaTeXDecoder(nn.Module):
    """Autoregressive decoder: (input_ids, encoder_output) → logits."""
    def __init__(self, vocab_size, max_seq_len=4096, decoder_dim=768, encoder_dim=768,
                 depth=12, num_heads=12, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.max_seq_len  = max_seq_len
        self.decoder_dim  = decoder_dim
        self.token_embed  = nn.Embedding(vocab_size, decoder_dim, padding_idx=0)
        self.pos_embed    = nn.Embedding(max_seq_len, decoder_dim)
        self.emb_drop     = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([DecoderBlock(decoder_dim, encoder_dim, num_heads,
                                                  mlp_ratio, dropout, max_seq_len)
                                     for _ in range(depth)])
        self.norm     = nn.LayerNorm(decoder_dim)
        self.lm_head  = nn.Linear(decoder_dim, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight   # weight tying
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.token_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight,   std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m is not self.lm_head:
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, input_ids, encoder_output, attention_mask=None):
        B, T = input_ids.shape
        pos  = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x    = self.emb_drop(self.token_embed(input_ids) + self.pos_embed(pos))
        pad_mask = (attention_mask == 0) if attention_mask is not None else None
        for blk in self.blocks:
            x = blk(x, encoder_output, tgt_key_padding_mask=pad_mask)
        return self.lm_head(self.norm(x))   # B × T × V

    @torch.no_grad()
    def generate(self, encoder_output, bos_id, eos_id, max_new_tokens=2048,
                 temperature=1.0, top_p=0.95, greedy=False):
        B, device = encoder_output.size(0), encoder_output.device
        generated = torch.full((B,1), bos_id, dtype=torch.long, device=device)
        finished  = torch.zeros(B, dtype=torch.bool, device=device)
        for _ in range(max_new_tokens):
            logits = self.forward(generated, encoder_output)
            next_t = logits[:,-1,:].argmax(-1, keepdim=True) if greedy \
                     else _sample_top_p(logits[:,-1,:], temperature, top_p)
            next_t = next_t.masked_fill(finished.unsqueeze(-1), eos_id)
            generated = torch.cat([generated, next_t], dim=1)
            finished  = finished | (next_t.squeeze(-1) == eos_id)
            if finished.all(): break
        return generated

def build_decoder(cfg):
    m = cfg["model"]
    return LaTeXDecoder(vocab_size=m["vocab_size"], max_seq_len=m["max_position_embeddings"],
                        decoder_dim=m["decoder_dim"], encoder_dim=m["encoder_dim"],
                        depth=m["decoder_layers"], num_heads=m["decoder_heads"],
                        mlp_ratio=m["decoder_mlp_ratio"], dropout=m["decoder_dropout"])

print("LaTeXDecoder defined ✓")


### 5.3 Full PDF2LaTeX Model

In [ ]:
class PDF2LaTeX(nn.Module):
    """Encoder-decoder model: PDF-page image → LaTeX token sequence."""
    def __init__(self, encoder, decoder, pad_id=0, label_smoothing=0.0):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        enc_dim = getattr(encoder, "embed_dim", None)
        dec_dim = decoder.decoder_dim
        self.enc_proj = nn.Linear(enc_dim, dec_dim) if (enc_dim and enc_dim != dec_dim) else nn.Identity()
        self.loss_fn  = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=label_smoothing)

    def forward(self, pixel_values, input_ids, attention_mask, labels):
        enc_out = self.enc_proj(self.encoder(pixel_values))
        logits  = self.decoder(input_ids, enc_out, attention_mask)
        loss    = self.loss_fn(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return loss, logits

    @torch.no_grad()
    def generate(self, pixel_values, bos_id, eos_id, max_new_tokens=2048,
                 temperature=1.0, top_p=0.95, greedy=False):
        enc_out = self.enc_proj(self.encoder(pixel_values))
        return self.decoder.generate(enc_out, bos_id, eos_id, max_new_tokens,
                                     temperature, top_p, greedy)

    def rollout_with_logprobs(self, pixel_values, bos_id, eos_id,
                              max_new_tokens=2048, temperature=1.0, top_p=0.95):
        """Sample a sequence and return per-token log-probs (for PPO/GRPO)."""
        enc_out   = self.enc_proj(self.encoder(pixel_values))
        B, device = enc_out.size(0), enc_out.device
        generated = torch.full((B,1), bos_id, dtype=torch.long, device=device)
        lp_list   = []
        finished  = torch.zeros(B, dtype=torch.bool, device=device)
        for _ in range(max_new_tokens):
            logits    = self.decoder(generated, enc_out)
            next_lgt  = logits[:,-1,:] / max(temperature, 1e-8)
            probs     = F.softmax(next_lgt, dim=-1)
            next_tok  = _sample_top_p(next_lgt, temperature=1.0, top_p=top_p)
            lp_list.append(torch.log(probs.gather(1, next_tok) + 1e-9))
            next_tok  = next_tok.masked_fill(finished.unsqueeze(-1), eos_id)
            generated = torch.cat([generated, next_tok], dim=1)
            finished  = finished | (next_tok.squeeze(-1) == eos_id)
            if finished.all(): break
        return generated, torch.cat(lp_list, dim=1)

    def compute_log_probs(self, pixel_values, input_ids, attention_mask=None):
        """Log-prob of each token in input_ids[1:] under current policy."""
        enc_out   = self.enc_proj(self.encoder(pixel_values))
        logits    = self.decoder(input_ids, enc_out, attention_mask)
        log_probs = F.log_softmax(logits, dim=-1)
        return log_probs[:,:-1,:].gather(2, input_ids[:,1:].unsqueeze(-1)).squeeze(-1)

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.state_dict(), str(path))

    def load(self, path, strict=True):
        self.load_state_dict(torch.load(str(path), map_location="cpu"), strict=strict)

def build_model(cfg, pad_id=0):
    return PDF2LaTeX(encoder=build_encoder(cfg), decoder=build_decoder(cfg),
                     pad_id=pad_id,
                     label_smoothing=cfg["training"].get("label_smoothing", 0.0))

# ── Quick sanity check (tiny model) ───────────────────────────────────────────
def _sanity_check_model():
    tiny_cfg = {
        "data":  {"image_size": 64, "max_seq_len": 32, "pdf_dpi": 72,
                  "train_dir": ".", "val_dir": ".", "num_workers": 0},
        "model": {"patch_size": 16, "encoder_dim": 64, "encoder_layers": 2,
                  "encoder_heads": 4, "encoder_mlp_ratio": 2.0, "encoder_dropout": 0.0,
                  "decoder_dim": 64, "decoder_layers": 2, "decoder_heads": 4,
                  "decoder_mlp_ratio": 2.0, "decoder_dropout": 0.0,
                  "vocab_size": 256, "max_position_embeddings": 32,
                  "vision_backbone": None, "init_from_pretrained": False},
        "training": {"label_smoothing": 0.0},
    }
    m = build_model(tiny_cfg, pad_id=PAD_ID).to(DEVICE)
    n = sum(p.numel() for p in m.parameters()) / 1e6
    print(f"Tiny model: {n:.2f} M params")

    B, T = 2, 16
    imgs  = torch.randn(B, 3, 64, 64, device=DEVICE)
    ids   = torch.randint(0, 256, (B, T), device=DEVICE)
    mask  = torch.ones(B, T, dtype=torch.long, device=DEVICE)
    lbl   = ids.clone(); lbl[:, -1] = -100
    loss, logits = m(imgs, ids, mask, lbl)
    print(f"Forward pass OK — loss={loss.item():.4f}, logits={logits.shape}")
    gen = m.generate(imgs[:1], BOS_ID, EOS_ID, max_new_tokens=8, greedy=True)
    print(f"Generation OK — shape={gen.shape}")

_sanity_check_model()


### 5.4 Reward Model (optional — for RLHF)

In [ ]:
class LaTeXTextEncoder(nn.Module):
    """Small bi-directional transformer to embed a LaTeX string."""
    def __init__(self, vocab_size, dim=256, depth=4, num_heads=8, max_seq_len=4096):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.pos_embed   = nn.Embedding(max_seq_len, dim)
        layer = nn.TransformerEncoderLayer(d_model=dim, nhead=num_heads,
                                           dim_feedforward=dim*4, dropout=0.1,
                                           activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)
    def forward(self, input_ids, attention_mask=None):
        B, T = input_ids.shape
        x = self.token_embed(input_ids) + self.pos_embed(torch.arange(T, device=input_ids.device).unsqueeze(0))
        pad_mask = (attention_mask == 0) if attention_mask is not None else None
        x = self.norm(self.transformer(x, src_key_padding_mask=pad_mask))
        if attention_mask is not None:
            msk = attention_mask.unsqueeze(-1).float()
            return (x * msk).sum(1) / msk.sum(1).clamp(min=1)
        return x.mean(1)

class RewardModel(nn.Module):
    """(image, latex) → scalar reward.  Trained with Bradley-Terry pairwise loss."""
    def __init__(self, vision_encoder, latex_encoder, encoder_dim=768, hidden_dim=512):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.latex_encoder  = latex_encoder
        self.fusion = nn.Sequential(
            nn.Linear(encoder_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(),
            nn.Linear(hidden_dim // 2, 1))
    def encode_image(self, pv):
        return self.vision_encoder(pv).mean(1)     # B × D
    def encode_latex(self, ids, mask=None):
        return self.latex_encoder(ids, mask)        # B × D
    def forward(self, pixel_values, input_ids, attention_mask=None):
        return self.fusion(torch.cat([self.encode_image(pixel_values),
                                      self.encode_latex(input_ids, attention_mask)], dim=-1)).squeeze(-1)
    def preference_loss(self, pv, chosen_ids, rejected_ids, chosen_mask=None, rejected_mask=None):
        return -F.logsigmoid(self.forward(pv, chosen_ids, chosen_mask) -
                             self.forward(pv, rejected_ids, rejected_mask)).mean()

def build_reward_model(cfg, vision_encoder):
    d = cfg["model"]["encoder_dim"]
    return RewardModel(vision_encoder,
                       LaTeXTextEncoder(cfg["model"]["vocab_size"], dim=d, depth=4,
                                        num_heads=8, max_seq_len=cfg["data"]["max_seq_len"]),
                       encoder_dim=d)

print("RewardModel defined ✓")


## 6 · Evaluation Metrics

In [ ]:
def token_accuracy(logits, labels):
    preds   = logits.argmax(dim=-1)
    mask    = labels != -100
    correct = ((preds == labels) & mask).sum().item()
    return correct / max(mask.sum().item(), 1)

def bleu_score(hypotheses, references, n=4):
    def ngrams(tokens, n):
        return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))
    bp_num, bp_den = 0, 0
    counts = [0]*n;  total = [0]*n
    for hyp, ref in zip(hypotheses, references):
        ht, rt = hyp.split(), ref.split()
        bp_num += len(ht);  bp_den += len(rt)
        for k in range(1, n+1):
            hng = ngrams(ht, k);  rng = ngrams(rt, k)
            counts[k-1] += sum(min(c, rng[g]) for g, c in hng.items())
            total[k-1]  += max(sum(hng.values()), 1)
    if min(counts) == 0: return 0.0
    log_avg = sum(math.log(c/t) for c,t in zip(counts, total)) / n
    bp = math.exp(min(0, 1 - bp_den / max(bp_num, 1)))
    return bp * math.exp(log_avg) * 100.0

def normalised_edit_distance(hyp, ref):
    try:
        import editdistance; dist = editdistance.eval(hyp, ref)
    except ImportError:
        # Pure-Python Levenshtein
        a, b = hyp, ref
        if len(a) < len(b): a, b = b, a
        if not b: return len(a) / max(len(hyp), len(ref), 1)
        prev = list(range(len(b)+1))
        for i, ca in enumerate(a, 1):
            curr = [i]
            for j, cb in enumerate(b, 1):
                curr.append(min(prev[j]+1, curr[j-1]+1, prev[j-1]+(ca!=cb)))
            prev = curr
        dist = prev[-1]
    return dist / max(len(hyp), len(ref), 1)

def format_score(latex_str):
    penalties = 0
    for ch_open, ch_close in [("{","}"), ("[","]")]:
        depth = 0
        for ch in latex_str:
            if ch == ch_open: depth += 1
            elif ch == ch_close: depth -= 1
            if depth < 0: penalties += 1; depth = 0
        if depth != 0: penalties += 1
    begins = re.findall(r"\\begin\{(\w+)\}", latex_str)
    ends   = re.findall(r"\\end\{(\w+)\}",   latex_str)
    if begins != ends: penalties += max(abs(len(begins)-len(ends)), 1)
    if len(latex_str.strip()) < 10: penalties += 3
    return max(0.0, 1.0 - penalties * 0.2)

def check_compilation(latex_str, compiler="pdflatex", timeout=30):
    body = latex_str if r"\begin{document}" in latex_str else (
        r"\documentclass{article}\n\begin{document}\n" + latex_str + r"\n\end{document}")
    with tempfile.TemporaryDirectory() as tmp:
        tex = os.path.join(tmp, "doc.tex")
        with open(tex, "w", encoding="utf-8") as f: f.write(body)
        try:
            r = subprocess.run([compiler, "-interaction=nonstopmode", "-halt-on-error",
                                "-output-directory", tmp, tex],
                               capture_output=True, timeout=timeout)
            return r.returncode == 0
        except (subprocess.TimeoutExpired, FileNotFoundError):
            return False

# ── Quick test ──────────────────────────────────────────────────────────────
hyps = [r"\textbf{Hello} $x^2$", r"\section{Intro}"]
refs = [r"\textbf{Hello} $x^2$", r"\section{Introduction}"]
print(f"BLEU:   {bleu_score(hyps, refs):.2f}")
print(f"EditD:  {normalised_edit_distance(hyps[0], refs[0]):.4f}")
print(f"Format: {format_score(hyps[0]):.2f}")


## 7 · Supervised Pre-training

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.cuda.amp import GradScaler, autocast

class CheckpointManager:
    def __init__(self, checkpoint_dir, keep_n=3):
        self.dir    = Path(checkpoint_dir)
        self.keep_n = keep_n
        self.dir.mkdir(parents=True, exist_ok=True)
    def save(self, model, optimizer=None, step=0, name=None):
        fname = name or f"ckpt_step_{step:07d}.pt"
        payload = {"step": step, "model": model.state_dict()}
        if optimizer: payload["optimizer"] = optimizer.state_dict()
        torch.save(payload, str(self.dir / fname))
        print(f"  ✓ Checkpoint saved → {self.dir / fname}")
        if not name: self._rotate()
    def _rotate(self):
        ckpts = sorted(glob.glob(str(self.dir / "ckpt_step_*.pt")),
                       key=os.path.getmtime)
        for old in ckpts[:-self.keep_n]: os.remove(old)
    @staticmethod
    def load(path, model, optimizer=None, device="cpu"):
        p = torch.load(path, map_location=device)
        model.load_state_dict(p["model"])
        if optimizer and "optimizer" in p: optimizer.load_state_dict(p["optimizer"])
        return p.get("step", 0)

def build_scheduler(optimizer, cfg, total_steps):
    wu = cfg["training"]["warmup_steps"]
    kind = cfg["training"]["lr_scheduler"]
    ws = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=wu)
    if kind == "cosine":
        ms = CosineAnnealingLR(optimizer, T_max=total_steps - wu, eta_min=1e-7)
    else:
        ms = LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=total_steps - wu)
    return SequentialLR(optimizer, schedulers=[ws, ms], milestones=[wu])

class Trainer:
    """Supervised pre-training loop."""
    def __init__(self, cfg):
        self.cfg    = cfg
        self.device = DEVICE
        self.model  = build_model(cfg, pad_id=PAD_ID).to(self.device)
        n_params = sum(p.numel() for p in self.model.parameters()) / 1e6
        print(f"Model: {n_params:.1f} M parameters")

        self.optimizer    = AdamW(self.model.parameters(),
                                  lr=cfg["training"]["learning_rate"],
                                  weight_decay=cfg["training"]["weight_decay"],
                                  betas=(0.9, 0.95))
        self.use_amp      = cfg["training"].get("bf16") or cfg["training"].get("fp16")
        self.amp_dtype    = torch.bfloat16 if cfg["training"].get("bf16") else torch.float16
        self.scaler       = GradScaler(enabled=cfg["training"].get("fp16", False))
        self.grad_accum   = cfg["training"]["gradient_accumulation_steps"]
        self.ckpt         = CheckpointManager(cfg["training"]["checkpoint_dir"],
                                              cfg["training"]["keep_last_n_checkpoints"])
        self.global_step  = 0
        self.best_val_loss = float("inf")
        self.scheduler    = None   # built after dataloaders

    def fit(self, train_loader, val_loader):
        total_steps = (len(train_loader) // self.grad_accum) * self.cfg["training"]["epochs"]
        self.scheduler = build_scheduler(self.optimizer, self.cfg, total_steps)
        print(f"Training for {self.cfg['training']['epochs']} epochs, "
              f"{total_steps} total steps")
        for epoch in range(self.cfg["training"]["epochs"]):
            self._train_epoch(epoch, train_loader)
            val_loss = self._validate(val_loader)
            print(f"[Epoch {epoch+1}] val_loss={val_loss:.4f}")
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.ckpt.save(self.model, self.optimizer, self.global_step, name="best.pt")
        print("Training complete ✓")

    def _train_epoch(self, epoch, loader):
        self.model.train()
        self.optimizer.zero_grad()
        for batch_idx, batch in enumerate(loader):
            pv  = batch["pixel_values"].to(self.device)
            ids = batch["input_ids"].to(self.device)
            msk = batch["attention_mask"].to(self.device)
            lbl = batch["labels"].to(self.device)
            with autocast(dtype=self.amp_dtype, enabled=self.use_amp):
                loss, logits = self.model(pv, ids, msk, lbl)
                loss = loss / self.grad_accum
            if self.cfg["training"].get("fp16"):
                self.scaler.scale(loss).backward()
            else:
                loss.backward()
            if (batch_idx + 1) % self.grad_accum == 0:
                if self.cfg["training"].get("fp16"):
                    self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(),
                                         self.cfg["training"]["max_grad_norm"])
                if self.cfg["training"].get("fp16"):
                    self.scaler.step(self.optimizer); self.scaler.update()
                else:
                    self.optimizer.step()
                self.scheduler.step()
                self.optimizer.zero_grad()
                self.global_step += 1
                if self.global_step % self.cfg["training"]["log_every_n_steps"] == 0:
                    acc = token_accuracy(logits.detach(), lbl)
                    lr  = self.scheduler.get_last_lr()[0]
                    print(f"  step={self.global_step:5d}  loss={loss.item()*self.grad_accum:.4f}"
                          f"  acc={acc:.3f}  lr={lr:.2e}")
                if self.global_step % self.cfg["training"]["save_every_n_steps"] == 0:
                    self.ckpt.save(self.model, self.optimizer, self.global_step)

    @torch.no_grad()
    def _validate(self, loader):
        self.model.eval()
        total, n = 0.0, 0
        for i, batch in enumerate(loader):
            if i >= self.cfg["training"]["val_max_batches"]: break
            pv  = batch["pixel_values"].to(self.device)
            ids = batch["input_ids"].to(self.device)
            msk = batch["attention_mask"].to(self.device)
            lbl = batch["labels"].to(self.device)
            with autocast(dtype=self.amp_dtype, enabled=self.use_amp):
                loss, _ = self.model(pv, ids, msk, lbl)
            total += loss.item(); n += 1
        return total / max(n, 1)

# ── How to run ────────────────────────────────────────────────────────────────
print("Trainer class ready. To start pre-training:")
print("  train_loader, val_loader = build_dataloaders(TRAIN_CFG, tokenizer)")
print("  trainer = Trainer(TRAIN_CFG)")
print("  trainer.fit(train_loader, val_loader)")


## 8 · Reward Functions

In [ ]:
import concurrent.futures

class CompilationReward:
    def __init__(self, compiler="pdflatex", timeout=30, n_workers=4):
        self.compiler = compiler; self.timeout = timeout
        self.executor = concurrent.futures.ThreadPoolExecutor(max_workers=n_workers)
    def __call__(self, latex_strings):
        futures = [self.executor.submit(check_compilation, s, self.compiler, self.timeout)
                   for s in latex_strings]
        return [float(f.result()) for f in futures]

class SimilarityReward:
    def __call__(self, hypotheses, references):
        return [1.0 - normalised_edit_distance(h, r) for h, r in zip(hypotheses, references)]

class FormatReward:
    def __call__(self, latex_strings):
        return [format_score(s) for s in latex_strings]

class HybridReward:
    """Weighted combination: compilation + similarity + format."""
    def __init__(self, compilation_weight=0.5, similarity_weight=0.3, format_weight=0.2,
                 compiler="pdflatex", compiler_timeout=30, n_workers=2):
        w = compilation_weight + similarity_weight + format_weight
        self.cw = compilation_weight / w
        self.sw = similarity_weight  / w
        self.fw = format_weight      / w
        self.comp_fn = CompilationReward(compiler, compiler_timeout, n_workers)
        self.sim_fn  = SimilarityReward()
        self.fmt_fn  = FormatReward()

    def __call__(self, hypotheses, references=None, **kwargs):
        B = len(hypotheses)
        rewards = [0.0] * B
        for i, r in enumerate(self.comp_fn(hypotheses)):
            rewards[i] += self.cw * r
        if references:
            for i, r in enumerate(self.sim_fn(hypotheses, references)):
                rewards[i] += self.sw * r
        for i, r in enumerate(self.fmt_fn(hypotheses)):
            rewards[i] += self.fw * r
        return rewards

def build_reward(cfg):
    rc = cfg["reward"]
    return HybridReward(rc["compilation_weight"], rc["similarity_weight"],
                        rc["format_weight"], rc["latex_compiler"], rc["compiler_timeout"])

# Quick test
reward_fn = HybridReward(compiler="pdflatex", n_workers=1)
test_latex = [r"\textbf{hello}", "not latex at all {{{{"]
# Note: compilation test requires pdflatex in PATH
fmt_scores = [format_score(s) for s in test_latex]
print("Format scores:", fmt_scores)
print("Similarity:", SimilarityReward()(test_latex, test_latex))
print("HybridReward (format+similarity only, skipping compilation):",
      [0.3 * 1.0 + 0.2 * f for f in fmt_scores])


## 9 · Rollout Engine

In [ ]:
from dataclasses import dataclass, field

@dataclass
class RolloutBuffer:
    pixel_values:    torch.Tensor
    generated_ids:   torch.Tensor
    attention_mask:  torch.Tensor
    old_log_probs:   torch.Tensor
    ref_log_probs:   object          # Optional[torch.Tensor]
    rewards:         torch.Tensor
    advantages:      torch.Tensor
    returns:         torch.Tensor
    tex_chunks:      list
    generated_texts: list

class RolloutEngine:
    """Collects rollouts from the current policy and scores them with the reward function."""
    def __init__(self, model, ref_model, reward_fn, tokenizer,
                 bos_id, eos_id, pad_id,
                 max_new_tokens=2048, temperature=0.8, top_p=0.95, num_samples=1):
        self.model = model; self.ref_model = ref_model; self.reward_fn = reward_fn
        self.tokenizer = tokenizer
        self.bos_id = bos_id; self.eos_id = eos_id; self.pad_id = pad_id
        self.max_new_tokens = max_new_tokens; self.temperature = temperature
        self.top_p = top_p; self.num_samples = num_samples

    @torch.no_grad()
    def collect(self, batch, device):
        pv         = batch["pixel_values"].to(device)
        tex_chunks = batch["tex_chunk"]
        buffers    = []
        for _ in range(self.num_samples):
            gen_ids, old_lp = self.model.rollout_with_logprobs(
                pv, self.bos_id, self.eos_id, self.max_new_tokens,
                self.temperature, self.top_p)
            hyps   = self._decode(gen_ids)
            rwds   = self.reward_fn(hypotheses=hyps, references=tex_chunks)
            rwd_t  = torch.tensor(rwds, dtype=torch.float32, device=device)
            attn_m = (gen_ids != self.pad_id).long()
            ref_lp = (self.ref_model.compute_log_probs(pv, gen_ids, attn_m)
                      if self.ref_model else None)
            buffers.append(RolloutBuffer(
                pixel_values=pv, generated_ids=gen_ids, attention_mask=attn_m,
                old_log_probs=old_lp, ref_log_probs=ref_lp,
                rewards=rwd_t, advantages=torch.zeros_like(rwd_t),
                returns=rwd_t.clone(), tex_chunks=tex_chunks, generated_texts=hyps))
        return buffers

    def _decode(self, generated_ids):
        texts = []
        for ids in generated_ids.tolist():
            if self.eos_id in ids: ids = ids[:ids.index(self.eos_id)]
            ids = [i for i in ids if i not in (self.bos_id, self.pad_id)]
            texts.append(self.tokenizer.decode(ids))
        return texts

print("RolloutEngine defined ✓")


## 10 · PPO Fine-tuning

In [ ]:
class ValueHead(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(input_dim, 256), nn.GELU(), nn.Linear(256, 1))
    def forward(self, encoder_output):
        return self.mlp(encoder_output.mean(1)).squeeze(-1)

class PPOTrainer:
    """Proximal Policy Optimisation fine-tuning loop."""
    def __init__(self, cfg, model, tokenizer, train_loader):
        self.cfg       = cfg
        self.device    = DEVICE
        self.model     = model.to(self.device)
        self.ref_model = copy.deepcopy(model).to(self.device)
        for p in self.ref_model.parameters(): p.requires_grad_(False)
        self.old_model = copy.deepcopy(model).to(self.device)
        for p in self.old_model.parameters(): p.requires_grad_(False)

        self.value_head   = ValueHead(model.encoder.embed_dim).to(self.device)
        self.reward_fn    = build_reward(cfg)
        self.rollout_eng  = RolloutEngine(
            self.old_model, self.ref_model, self.reward_fn, tokenizer,
            BOS_ID, EOS_ID, PAD_ID,
            max_new_tokens=cfg["rollout"]["max_new_tokens"],
            temperature=cfg["rollout"]["temperature"],
            top_p=cfg["rollout"]["top_p"], num_samples=1)

        tc = cfg["training"];  pc = cfg["ppo"]
        self.optim       = AdamW(list(model.parameters()) + list(self.value_head.parameters()),
                                 lr=tc["policy_lr"], weight_decay=tc["weight_decay"])
        self.clip_eps    = pc["clip_epsilon"]
        self.val_coef    = pc["value_loss_coef"]
        self.ent_coef    = pc["entropy_coef"]
        self.kl_coef     = pc["kl_coef"]
        self.target_kl   = pc["target_kl"]
        self.ppo_epochs  = pc["epochs"]
        self.max_grad    = tc["max_grad_norm"]
        self.train_loader = train_loader
        self.ckpt        = CheckpointManager(tc["checkpoint_dir"])
        self.global_step = 0

    def train(self, total_steps=None):
        total_steps = total_steps or self.cfg["training"]["total_steps"]
        data_iter = iter(self.train_loader)
        for step in range(total_steps):
            try: batch = next(data_iter)
            except StopIteration: data_iter = iter(self.train_loader); batch = next(data_iter)

            buffers = self.rollout_eng.collect(batch, self.device)
            buf     = self._compute_advantages(buffers[0])
            metrics = self._ppo_update(buf)
            self.old_model.load_state_dict(self.model.state_dict())
            self.global_step += 1

            if self.global_step % self.cfg["training"]["log_every_n_steps"] == 0:
                print(f"[PPO step {self.global_step}] "
                      f"reward={buf.rewards.mean():.3f}  "
                      f"policy_loss={metrics.get('policy_loss', 0):.4f}  "
                      f"kl={metrics.get('kl', 0):.5f}")
            if self.global_step % self.cfg["training"]["save_every_n_steps"] == 0:
                self.ckpt.save(self.model, self.optim, self.global_step)

    def _compute_advantages(self, buf):
        with torch.no_grad():
            enc_out = self.model.encoder(buf.pixel_values)
            values  = self.value_head(enc_out)
        adv = buf.rewards - values
        buf.advantages = (adv - adv.mean()) / (adv.std() + 1e-8)
        return buf

    def _ppo_update(self, buf):
        metrics = {}
        for epoch in range(self.ppo_epochs):
            new_lp = self.model.compute_log_probs(buf.pixel_values, buf.generated_ids, buf.attention_mask)
            mask   = buf.attention_mask[:, 1:].float()
            ratio  = torch.exp((new_lp * mask).sum(-1) - (buf.old_log_probs * mask).sum(-1).detach())
            adv    = buf.advantages.detach()
            surr   = -torch.min(ratio*adv, torch.clamp(ratio,1-self.clip_eps,1+self.clip_eps)*adv).mean()
            enc_out = self.model.encoder(buf.pixel_values)
            vl     = F.mse_loss(self.value_head(enc_out), buf.returns.detach())
            kl     = ((new_lp - buf.ref_log_probs)*mask).sum(-1).mean() if buf.ref_log_probs is not None else torch.zeros(1, device=self.device)
            ent    = -(new_lp * mask).sum(-1).mean()
            loss   = surr + self.val_coef*vl + self.kl_coef*kl - self.ent_coef*ent
            self.optim.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(list(self.model.parameters()) + list(self.value_head.parameters()), self.max_grad)
            self.optim.step()
            metrics = {"policy_loss": surr.item(), "value_loss": vl.item(), "kl": kl.item() if hasattr(kl, 'item') else 0}
            if kl.item() > 2 * self.target_kl: break
        return metrics

print("PPOTrainer defined ✓")
print("Usage:")
print("  ppo = PPOTrainer(FINETUNE_CFG, model, tokenizer, train_loader)")
print("  ppo.train(total_steps=5000)")


## 11 · GRPO Fine-tuning (recommended)

GRPO samples **G** candidates per prompt, normalises rewards within the group, and skips the critic. Simpler than PPO and often more stable.

In [ ]:
class GRPOTrainer:
    """Group Relative Policy Optimisation (DeepSeek-R1 style)."""
    def __init__(self, cfg, model, tokenizer, train_loader):
        self.cfg    = cfg
        self.device = DEVICE
        self.model  = model.to(self.device)

        self.ref_model = copy.deepcopy(model).to(self.device)
        for p in self.ref_model.parameters(): p.requires_grad_(False)
        self.old_model = copy.deepcopy(model).to(self.device)
        for p in self.old_model.parameters(): p.requires_grad_(False)

        self.G         = cfg["rollout"]["num_samples_per_prompt"]
        self.reward_fn = build_reward(cfg)
        self.rollout   = RolloutEngine(
            self.old_model, self.ref_model, self.reward_fn, tokenizer,
            BOS_ID, EOS_ID, PAD_ID,
            max_new_tokens=cfg["rollout"]["max_new_tokens"],
            temperature=cfg["rollout"]["temperature"],
            top_p=cfg["rollout"]["top_p"], num_samples=self.G)

        tc = cfg["training"];  gc = cfg["grpo"]
        self.optim       = AdamW(model.parameters(), lr=tc["policy_lr"],
                                 weight_decay=tc["weight_decay"])
        self.clip_eps    = gc["clip_epsilon"]
        self.kl_coef     = gc["kl_coef"]
        self.norm_adv    = gc["normalize_advantages"]
        self.max_grad    = tc["max_grad_norm"]
        self.train_loader = train_loader
        self.ckpt        = CheckpointManager(tc["checkpoint_dir"])
        self.global_step = 0

    def train(self, total_steps=None):
        total_steps = total_steps or self.cfg["training"]["total_steps"]
        data_iter = iter(self.train_loader)
        print(f"Starting GRPO (G={self.G}) for {total_steps} steps …")
        for step in range(total_steps):
            try: batch = next(data_iter)
            except StopIteration: data_iter = iter(self.train_loader); batch = next(data_iter)

            # Collect G rollouts per prompt
            buffers = self.rollout.collect(batch, self.device)

            # Group-relative advantage: A_i = (R_i - mean_G) / std_G
            rwd_stack = torch.stack([b.rewards for b in buffers])  # G × B
            g_mean    = rwd_stack.mean(0, keepdim=True)
            g_std     = rwd_stack.std(0, keepdim=True).clamp(min=1e-8)
            for buf in buffers:
                buf.advantages = (buf.rewards - g_mean.squeeze(0)) / g_std.squeeze(0)

            metrics = self._grpo_update(buffers)
            self.old_model.load_state_dict(self.model.state_dict())
            self.global_step += 1

            if self.global_step % self.cfg["training"]["log_every_n_steps"] == 0:
                mean_rwd = rwd_stack.mean().item()
                print(f"[GRPO step {self.global_step}] "
                      f"mean_reward={mean_rwd:.3f}  "
                      f"loss={metrics.get('loss', 0):.4f}  "
                      f"kl={metrics.get('kl', 0):.5f}")
            if self.global_step % self.cfg["training"]["save_every_n_steps"] == 0:
                self.ckpt.save(self.model, self.optim, self.global_step)
        print("GRPO fine-tuning complete ✓")

    def _grpo_update(self, buffers):
        total_loss = 0.0; total_kl = 0.0
        self.optim.zero_grad()
        for buf in buffers:
            new_lp = self.model.compute_log_probs(buf.pixel_values, buf.generated_ids, buf.attention_mask)
            mask   = buf.attention_mask[:, 1:].float()
            ratio  = torch.exp((new_lp * mask).sum(-1) - (buf.old_log_probs * mask).sum(-1).detach())
            adv    = buf.advantages.detach()
            surr   = -torch.min(ratio*adv, torch.clamp(ratio,1-self.clip_eps,1+self.clip_eps)*adv).mean()
            kl = ((new_lp - buf.ref_log_probs)*mask).sum(-1).mean() if buf.ref_log_probs is not None else torch.zeros(1, device=self.device)
            loss   = (surr + self.kl_coef * kl) / len(buffers)
            loss.backward()
            total_loss += loss.item(); total_kl += kl.item() if hasattr(kl, 'item') else 0
        nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad)
        self.optim.step()
        return {"loss": total_loss, "kl": total_kl / len(buffers)}

print("GRPOTrainer defined ✓")
print("Usage:")
print("  grpo = GRPOTrainer(FINETUNE_CFG, model, tokenizer, train_loader)")
print("  grpo.train(total_steps=5000)")


## 12 · Reward Model Training (Optional)

Train a learned reward model on human preference pairs `{pdf, chosen_latex, rejected_latex}`.
Data format — a JSONL file with one JSON object per line:
```json
{"pdf": "data/train/sample_001.pdf", "chosen": "\\textbf{good}", "rejected": "bad{{"}
```


In [ ]:
from torch.utils.data import Dataset as TorchDataset

class PreferenceDataset(TorchDataset):
    def __init__(self, jsonl_path, tokenizer, image_transform, max_seq_len=2048, dpi=150):
        self.records = [json.loads(l) for l in open(jsonl_path)]
        self.tokenizer = tokenizer; self.image_transform = image_transform
        self.max_seq_len = max_seq_len; self.dpi = dpi
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        r = self.records[idx]
        pv = self.image_transform(render_pdf_page(r["pdf"], dpi=self.dpi)).float()
        ci, cm = self._tok(r["chosen"])
        ri, rm = self._tok(r["rejected"])
        return {"pixel_values": pv, "chosen_ids": ci, "chosen_mask": cm,
                "rejected_ids": ri, "rejected_mask": rm}
    def _tok(self, text):
        ids = self.tokenizer.encode(text).ids[:self.max_seq_len]
        pad = self.max_seq_len - len(ids)
        return (torch.tensor(ids + [PAD_ID]*pad, dtype=torch.long),
                torch.tensor([1]*len(ids) + [0]*pad, dtype=torch.long))

def train_reward_model(cfg, pref_jsonl, output_dir="checkpoints/reward_model",
                       epochs=5, vision_encoder=None):
    """Fine-tune a reward model on human preference data."""
    if vision_encoder is None:
        vision_encoder = build_encoder(cfg)
    rm       = build_reward_model(cfg, vision_encoder).to(DEVICE)
    img_tf   = build_image_transform(cfg["data"]["image_size"])
    ds       = PreferenceDataset(pref_jsonl, tokenizer, img_tf,
                                 max_seq_len=cfg["data"]["max_seq_len"])
    loader   = DataLoader(ds, batch_size=4, shuffle=True, num_workers=0)
    optim    = AdamW(rm.parameters(), lr=1e-4, weight_decay=0.01)
    ckpt     = CheckpointManager(output_dir)
    step     = 0

    for epoch in range(epochs):
        if epoch == epochs // 2:
            print("Unfreezing vision encoder")
            for p in rm.vision_encoder.parameters(): p.requires_grad_(True)
        for batch in loader:
            pv  = batch["pixel_values"].to(DEVICE)
            ci  = batch["chosen_ids"].to(DEVICE);   cm = batch["chosen_mask"].to(DEVICE)
            ri  = batch["rejected_ids"].to(DEVICE); rm_ = batch["rejected_mask"].to(DEVICE)
            loss = rm.preference_loss(pv, ci, ri, cm, rm_)
            optim.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(rm.parameters(), 1.0); optim.step()
            step += 1
            if step % 10 == 0:
                print(f"  [RM epoch {epoch+1} step {step}] loss={loss.item():.4f}")
    ckpt.save(rm, optim, step, name="best.pt")
    print("Reward model training complete ✓")
    return rm

print("train_reward_model() ready.")
print("Usage:")
print('  rm = train_reward_model(TRAIN_CFG, "data/preferences.jsonl")')


## 13 · Inference — Run on New PDFs

In [ ]:
@torch.no_grad()
def pdf_to_latex(pdf_path, model, image_transform, device,
                 max_new_tokens=3072, temperature=0.2, top_p=0.9, greedy=False):
    """Convert all pages of a PDF to LaTeX strings.  Returns list[str]."""
    n_pages = count_pdf_pages(pdf_path)
    results = []
    for page_idx in range(n_pages):
        img = render_pdf_page(pdf_path, page_idx)
        pv  = image_transform(img).unsqueeze(0).to(device)
        gen = model.generate(pv, BOS_ID, EOS_ID, max_new_tokens, temperature, top_p, greedy)
        ids = gen[0].tolist()
        if EOS_ID in ids: ids = ids[:ids.index(EOS_ID)]
        ids = [i for i in ids if i != BOS_ID and i != PAD_ID]
        results.append(tokenizer.decode(ids))
    return results

def assemble_document(page_chunks):
    """Stitch page chunks into a compilable LaTeX document."""
    if not page_chunks: return ""
    if r"\documentclass" in page_chunks[0]:
        first = page_chunks[0]; rest = r"\newpage".join(page_chunks[1:])
        if r"\end{document}" in first:
            return first.replace(r"\end{document}", rest + "\n" + r"\end{document}")
        return first + "\n" + rest + "\n" + r"\end{document}"
    body = r"\newpage".join(page_chunks)
    return (r"\documentclass{article}\n\usepackage{amsmath,amssymb,graphicx}\n"
            r"\begin{document}\n" + body + "\n" + r"\end{document}")

def run_inference(pdf_path, model, output_path=None, **gen_kwargs):
    """High-level wrapper: PDF → .tex file."""
    img_tf  = build_image_transform(TRAIN_CFG["data"]["image_size"])
    chunks  = pdf_to_latex(pdf_path, model, img_tf, DEVICE, **gen_kwargs)
    doc     = assemble_document(chunks)
    if output_path:
        Path(output_path).write_text(doc, encoding="utf-8")
        print(f"Written → {output_path}")
    return doc

print("run_inference() ready.")
print("Usage:")
print('  latex = run_inference("paper.pdf", model, output_path="paper.tex")')


## 14 · Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, val_loader, max_batches=50, compiler="none"):
    """
    Compute token accuracy, BLEU-4, edit distance, format score,
    and optionally compilation rate.
    """
    model.eval()
    img_tf     = build_image_transform(TRAIN_CFG["data"]["image_size"])
    acc_total, n_batches = 0.0, 0
    hypotheses, references = [], []

    for i, batch in enumerate(val_loader):
        if i >= max_batches: break
        pv  = batch["pixel_values"].to(DEVICE)
        ids = batch["input_ids"].to(DEVICE)
        msk = batch["attention_mask"].to(DEVICE)
        lbl = batch["labels"].to(DEVICE)
        loss, logits = model(pv, ids, msk, lbl)
        acc_total += token_accuracy(logits, lbl)

        gen = model.generate(pv, BOS_ID, EOS_ID, max_new_tokens=256, greedy=True)
        for g in gen.tolist():
            if EOS_ID in g: g = g[:g.index(EOS_ID)]
            hypotheses.append(tokenizer.decode([t for t in g if t not in (BOS_ID, PAD_ID)]))
        for c in batch["tex_chunk"]:
            references.append(c)
        n_batches += 1

    results = {
        "token_accuracy": acc_total / max(n_batches, 1),
        "bleu4":          bleu_score(hypotheses, references),
        "edit_distance":  sum(normalised_edit_distance(h,r) for h,r in zip(hypotheses,references))
                          / max(len(hypotheses), 1),
        "format_score":   sum(format_score(h) for h in hypotheses) / max(len(hypotheses), 1),
    }
    if compiler and compiler.lower() != "none":
        print("Checking compilation rates …")
        results["compilation_rate"] = sum(
            check_compilation(h, compiler) for h in hypotheses[:20]) / min(20, len(hypotheses))

    print("\n── Evaluation Results ────────────────")
    for k, v in results.items():
        print(f"  {k:<22}  {v:.4f}")
    return results

print("evaluate() ready.")
print("Usage:")
print("  results = evaluate(model, val_loader, max_batches=20)")


## 15 · End-to-End Pipeline Summary

Run all steps in order once your data directory is populated:

In [ ]:
# ┌─────────────────────────────────────────────────────────┐
# │  Full pipeline — uncomment and run step-by-step         │
# └─────────────────────────────────────────────────────────┘

# STEP 1 — Train tokeniser (once)
# tokenizer = train_tokenizer_from_corpus("data/train", vocab_size=8192,
#                                          output_path="tokenizer.json")

# STEP 2 — (Optional) Pre-render PDFs for faster training
# prerender_dataset("data/train", "data/train_cache")

# STEP 3 — Build dataloaders
# train_loader, val_loader = build_dataloaders(TRAIN_CFG, tokenizer)

# STEP 4 — Build model
# model = build_model(TRAIN_CFG, pad_id=PAD_ID)

# STEP 5 — Supervised pre-training
# trainer = Trainer(TRAIN_CFG)
# trainer.fit(train_loader, val_loader)

# STEP 6 — Evaluate baseline
# results = evaluate(model, val_loader, max_batches=20)

# STEP 7a — GRPO fine-tuning (recommended)
# grpo = GRPOTrainer(FINETUNE_CFG, model, tokenizer, train_loader)
# grpo.train(total_steps=5000)

# STEP 7b — Or PPO fine-tuning
# ppo = PPOTrainer(FINETUNE_CFG, model, tokenizer, train_loader)
# ppo.train(total_steps=5000)

# STEP 8 — (Optional) Train reward model on human prefs, then re-run GRPO
# rm = train_reward_model(TRAIN_CFG, "data/preferences.jsonl")

# STEP 9 — Inference on a new PDF
# latex = run_inference("paper.pdf", model, output_path="paper.tex")

# STEP 10 — Final evaluation
# results = evaluate(model, val_loader, max_batches=50, compiler="pdflatex")

print("""
Pipeline components loaded:
  ✓ Tokeniser        build_latex_tokenizer / train_tokenizer_from_corpus
  ✓ PDF rendering    render_pdf_page / count_pdf_pages / prerender_dataset
  ✓ Dataset          PDFLatexDataset / build_dataloaders
  ✓ Encoder          VisionEncoder / HFVisionEncoder / build_encoder
  ✓ Decoder          LaTeXDecoder / build_decoder
  ✓ Full model       PDF2LaTeX / build_model
  ✓ Reward model     RewardModel / build_reward_model / train_reward_model
  ✓ Metrics          token_accuracy / bleu_score / format_score / check_compilation
  ✓ Trainer          Trainer (supervised)
  ✓ Rewards          HybridReward / CompilationReward / SimilarityReward / FormatReward
  ✓ Rollout          RolloutEngine / RolloutBuffer
  ✓ PPO              PPOTrainer
  ✓ GRPO             GRPOTrainer
  ✓ Inference        pdf_to_latex / run_inference
  ✓ Evaluation       evaluate
""")
